[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MeteoSwiss/opendata-nwp-demos/blob/main/09_constant_parameters.ipynb)
[![launch - renku](https://renkulab.io/renku-badge.svg)](https://renkulab.io/p/meteoswiss/opendata-nwp-demos/sessions/01KME52HC2FZ6ZHB30SSFG08PW/start)

# Accessing constant parameters

This notebook demonstrates the retrieval of **constant run parameters** such as grid definition, vertical layers, and surface descriptors that are not part of the forecast GRIB files, and shows how to verify grid consistency with forecast parameters. The data is provided by MeteoSwiss as part of Switzerland’s [Open Government Data (OGD) initiative](https://www.meteoswiss.admin.ch/services-and-publications/service/open-data.html).



---

## 🔍 **What You’ll Do in This Notebook**

 📥  **Retrieve vertical constants**  
    The parameter HHL is the only vertical constant; retrieval is shown for ICON-CH1-EPS and ICON-CH2-EPS.
    
| Name      | Description                                          |
|-----------|------------------------------------------------------|
| HHL       | Geometric height of the layer limits above sea level (for more information see [vertical grid](https://opendatadocs.meteoswiss.ch/e-forecast-data/e2-e3-numerical-weather-forecasting-model#vertical-grid)) |

 📥  **Retrieve horizontal constants**  
    The available horizontal constant parameters are CLON, CLAT, DEPTH_L, FR_ICE, FR_LAKE, FR_LAND, FOR_D, HSURF, LAI, PLCOV, ROOTDP, SKC, SOILTYP, SSO_GAMMA, SSO_SIGMA, SSO_STDH and SSO_THETA for ICON-CH1-EPS and ICON-CH2-EPS.
    
| Name      | Description                                                                      |
|-----------|----------------------------------------------------------------------------------|
| CLON      | Longitude of the center point coordinate of each triangle on the horizontal grid |
| CLAT      | Latitude of the center point coordinate of each triangle on the horizontal grid  |
| FR_LAND   | Land cover indicator (1 for land, 0 for sea)                                     |
| HSURF     | Geometric height of the earth's surface above sea level                          |
| SOILTYPE  | Type of soil, based on the [soilType.table](https://github.com/COSMO-ORG/eccodes-cosmo-resources/blob/master/definitions/soilType.table) |
| DEPTH_LK  | Lake depth                                                                       |
| FOR_D     | Deciduous forest fraction                                                        |
| FR_ICE    | Sea ice fraction                                                                 |
| FR_LAKE   | Lake fraction                                                                    |
| LAI       | Leaf area index                                                                  |
| PLCOV     | Plant cover fraction                                                             |
| ROOTDP    | Root depth                                                                       |
| SKC       | Skin conductivity                                                                |
| SSO_GAMMA | Sub-grid scale orography anisotropy                                              |
| SSO_SIGMA | Sub-grid scale orography slope                                                   |
| SSO_STDH  | Sub-grid scale orography standard deviation                                      |
| SSO_THETA | Sub-grid scale orography direction                                               |



 ✅  **Verify**  
    Using GRIB metadata (uuidOfHGrid), you'll verify grid consistency with a forecast parameter from ICON-CH2-EPS.
    
 📈  **Plot**  
    Create a profile of the vertical wind speed at Zurich airport, and show mean precipitation as a function of surface height.

---

> ⚠️ **Warning**: The constant parameters are only considered constant per run! Although they are not expected to change often, we recommend that you check that the 'uuidOfHGrid' agrees with the forecast data, especially if you cache the constants.

> ⚠️ **Warning**: At the moment, the collection assets are not versioned such that there is no way to fetch a previous version of the constant run parameters. If you are building an archive, you may need to keep track of those constants.

⚙️ Notebook Setup

This cell installs the required dependencies when running the notebook in Google Colab or RenkuLab.

It is skipped in a local Jupyter environment, where dependencies are assumed to be installed already.

In [ ]:
# 📦 Notebook setup: Colab + RenkuLab
import sys, os, pathlib

IN_COLAB = "google.colab" in sys.modules
IN_RENKU = "RENKU_BASE_URL" in os.environ or "RENKU_BASE_URL_PATH" in os.environ

if IN_COLAB:
    !git clone https://github.com/MeteoSwiss/opendata-nwp-demos.git
    %cd opendata-nwp-demos

if IN_COLAB or IN_RENKU:
    !pip install poetry && poetry config virtualenvs.in-project true && poetry install --no-ansi

    venv = pathlib.Path(".venv")
    site = venv / "lib" / f"python{sys.version_info.major}.{sys.version_info.minor}" / "site-packages"
    sys.path.insert(0, str(site))
    os.environ["ECCODES_DEFINITION_PATH"] = str((venv / "share/eccodes-cosmo-resources/definitions").resolve())

⚙️ Definition Setup

MeteoSwiss uses custom ecCodes definitions for ICON data. To correctly interpret MeteoSwiss-specific `shortName` values, you must configure the definition path by running the cell below.

In [ ]:
import os
import eccodes_cosmo_resources

os.environ["ECCODES_DEFINITION_PATH"] = str(
    eccodes_cosmo_resources.get_definitions_path()
)

os.environ["ECCODES_VERSION_CHECK_OFF"] = "1"

## 📥 Retrieve constant parameters
The constant parameters are separated into horizontal and vertical constants, and available for both the ICON-CH1-EPS and ICON-CH2-EPS collections.

Both types of constants can be retrieved using the `meteoswiss-opendata-constants` plugin.

In [ ]:
import earthkit.data as ekd

horizontal_ch2 = ekd.from_source(
    "meteoswiss-opendata-constants",
    collection="ogd-forecasting-icon-ch2",
    asset="horizontal",
)

vertical_ch2 = ekd.from_source(
    "meteoswiss-opendata-constants",
    collection="ogd-forecasting-icon-ch2",
    asset="vertical",
)

The only vertical constant is HHL, referring to the geometric height of the layer limits above sea level. For more information, see [model grid and static data](https://opendatadocs.meteoswiss.ch/e-forecast-data/e2-e3-numerical-weather-forecasting-model#model-grid-and-static-data).

> 💡 **Tip**: The HHL parameter is also used in the notebook [05_interpolate_vertically.ipynb](05_interpolate_vertically.ipynb).

In [ ]:
vertical_fl_ch2 = vertical_ch2.to_fieldlist()
vertical_fl_ch2.ls()

The following horizontal constants are available.

> 💡 **Tip**: For more details consult [model grid and static data](https://opendatadocs.meteoswiss.ch/e-forecast-data/e2-e3-numerical-weather-forecasting-model#model-grid-and-static-data).

| Name      | Description                                                                      |
|-----------|----------------------------------------------------------------------------------|
| CLON      | Longitude of the center point coordinate of each triangle on the horizontal grid |
| CLAT      | Latitude of the center point coordinate of each triangle on the horizontal grid  |
| FR_LAND   | Land cover indicator (1 for land, 0 for sea)                                     |
| HSURF     | Geometric height of the earth's surface above sea level                          |
| SOILTYPE  | Type of soil, based on the [soilType.table](https://github.com/COSMO-ORG/eccodes-cosmo-resources/blob/master/definitions/soilType.table) |
| DEPTH_LK  | Lake depth                                                                       |
| FOR_D     | Deciduous forest fraction                                                        |
| FR_ICE    | Sea ice fraction                                                                 |
| FR_LAKE   | Lake fraction                                                                    |
| LAI       | Leaf area index                                                                  |
| PLCOV     | Plant cover fraction                                                             |
| ROOTDP    | Root depth                                                                       |
| SKC       | Skin conductivity                                                                |
| SSO_GAMMA | Sub-grid scale orography anisotropy                                              |
| SSO_SIGMA | Sub-grid scale orography slope                                                   |
| SSO_STDH  | Sub-grid scale orography standard deviation                                      |
| SSO_THETA | Sub-grid scale orography direction                                               |


In [ ]:
horizontal_fl_ch2 = horizontal_ch2.to_fieldlist()
horizontal_fl_ch2.ls()

## ✅ Verify grid consistency

The grib metadata carries a lot of information:

In [ ]:
# show all metadata
hhl_ch2 = vertical_fl_ch2[0]

A universally unique identifier (UUID) of the horizontal grid the data is associated with is provided. You can see that the ICON-CH1-EPS and ICON-CH2-EPS models use a different grid:

In [ ]:
hhl_ch2.metadata("uuidOfHGrid")

In [ ]:
vertical_fl_ch1 = ekd.from_source(
    "meteoswiss-opendata-constants",
    collection="ogd-forecasting-icon-ch1",
    asset="vertical",
).to_fieldlist()

print("CH1-EPS HHL    : ", vertical_fl_ch1[0].metadata("uuidOfHGrid"))
print("CH2-EPS HHL    : ", hhl_ch2.metadata("uuidOfHGrid"))
print("CH2-EPS DEPTH_LK: ", horizontal_fl_ch2[0].metadata("uuidOfHGrid"))

We fetch now the vertical wind speed from the ICON-CH2-EPS control forecast, and compare the UUID of the grid with the UUID obtained from the vertical constants.

> ⚠️ **Warning**: The constant parameters are only considered constant per run! Although they are not expected to change often, we recommend that you check that the 'uuidOfHGrid' agrees with the forecast data, especially if you cache the constants.

In [ ]:
import earthkit.data as ekd
ekd.config.set("cache-policy", "temporary")

# Create request
request = {
    "collection": "ogd-forecasting-icon-ch2",
    "variable": "W",
    "ref_time": "latest",
    "perturbed": False,
    "lead_time": "P3DT10H"
}

# Fetch the data
forecast = ekd.from_source(
    "meteoswiss-opendata",
    **request,
    )

# Convert to fieldlist
vertical_wind_fl = forecast.to_fieldlist()

# Convert to xarray
ds_vertical_wind = forecast.to_xarray(
    time_dims=["forecast_reference_time", "step"],
    squeeze=False
    )

Compare the UUIDs:

In [ ]:
assert vertical_wind_fl[0].metadata("uuidOfHGrid") == hhl_ch2.metadata("uuidOfHGrid")

## 📈 Create a profile of the vertical wind speed

Now that we have verified consistency of the constant run parameters (HHL) and forecast data (W), we can create a plot.

In [ ]:
from earthkit.geo import nearest_point_haversine

ds_hhl_ch2 = vertical_ch2.to_xarray(
    time_dims=["forecast_reference_time", "step"],
    squeeze=False
    )

# target coordinates
zrh_geo_point = (47.453928, 8.565074)  # zrh airport

# get index of closest cell
p = nearest_point_haversine(zrh_geo_point, (ds_vertical_wind.latitude.values, ds_vertical_wind.longitude.values))[0][0]

# extract the data to plot at index p
w_profile = ds_vertical_wind.W.isel(values=p, member=0, forecast_reference_time=0, step=0)
h_profile = ds_hhl_ch2.HHL.isel(values=p, member=0, forecast_reference_time=0, step=0)

In [ ]:
from earthkit.plots import Figure

# create plot

fig = Figure(rows=1, columns=1, size=(6,4))
subfig = fig.add_subplot()

subfig.line(x=w_profile, y=h_profile)
subfig.ax.axvline(x=0, color='gray', linewidth=0.8)

subfig.ax.set_xlabel(f"Vertical Wind ({ds_vertical_wind.W.units})")
subfig.ax.set_ylabel(f"Height above sea level ({ds_hhl_ch2.HHL.units})")

subfig.ax.set_title(f"Vertical Profile at Zurich Airport: Vertical Wind", loc='left')

fig.show()

## 📈 Show the precipitation amounts against surface height

In [ ]:
# fetch total precipitation for 5 days lead time
import earthkit.data as ekd
ekd.config.set("cache-policy", "temporary")

# Create request
request = {
    "collection": "ogd-forecasting-icon-ch2",
    "variable": "TOT_PREC",
    "ref_time": "latest",
    "perturbed": False,
    "lead_time": "P5DT00H"
}

# Fetch the data
forecast = ekd.from_source(
    "meteoswiss-opendata",
    **request,
    )

# Convert to fieldlist
total_precipitation_fl = forecast.to_fieldlist()

# Convert to xarray
ds_total_precipitation = forecast.to_xarray(
    time_dims=["forecast_reference_time", "step"],
    squeeze=False
    )

In [ ]:
# verify consistency of the grid
hsurf_ch2 = horizontal_fl_ch2[4]
assert total_precipitation_fl[0].metadata("uuidOfHGrid") == hsurf_ch2.metadata("uuidOfHGrid")

In [ ]:
import numpy as np
import pandas as pd
from earthkit.plots import Figure

# convert to xarray
ds_hsurf_ch2 = hsurf_ch2.to_xarray(
    time_dims=["forecast_reference_time", "step"],
    squeeze=False
)

# squeeze to 1D arrays over cells
prec_vals = ds_total_precipitation.TOT_PREC.squeeze().values
hsurf_vals = ds_hsurf_ch2.HSURF.squeeze().values

# bin cells by surface height and compute mean precipitation per bin
n_bins = 20
bins = np.linspace(hsurf_vals.min(), hsurf_vals.max(), n_bins + 1)
bin_centers = 0.5 * (bins[:-1] + bins[1:])

df = pd.DataFrame({"prec": prec_vals, "hsurf": hsurf_vals})
df["bin"] = pd.cut(df["hsurf"], bins=bins, labels=bin_centers, include_lowest=True)
mean_prec = df.groupby("bin", observed=True)["prec"].mean()

ref_time = pd.to_datetime(ds_total_precipitation.forecast_reference_time.values[0])
lead_time = ds_total_precipitation.step.values.astype('timedelta64[h]')[0]

# create plot
fig = Figure(rows=1, columns=1, size=(6,4))
subfig = fig.add_subplot()

subfig.line(x=mean_prec.index.astype(float).values, y=mean_prec.values)

subfig.ax.set_xlabel(f"Surface Height ({ds_hsurf_ch2.HSURF.units})")
subfig.ax.set_ylabel(f"Mean Precipitation ({ds_total_precipitation.TOT_PREC.units})")
subfig.ax.set_title(f"Mean Precipitation vs Surface Height, {ref_time} UTC +{lead_time}", loc='left')
subfig.ax.set_ylim(bottom=-0.1)

fig.show()